# Create Combined Table: Spark + DAX Guide
## Step-by-step guide to combine two tables using Spark and generate DAX measures

This notebook demonstrates:
1. Loading two source tables (SalesPerson and Sales)
2. Combining them using Spark join operations
3. Creating a new table with calculated fields
4. Defining DAX measures for analytics
5. Saving to production-ready format

## Section 1: Import Required Libraries
Import PySpark, Spark SQL functions, and utilities for data processing

In [ ]:
# Import PySpark libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, concat_ws, upper, lower, trim,
    count, sum, avg, max, min, round,
    datediff, to_date, year, month, dayofweek
)
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
import pandas as pd

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("CombinedTableCreation") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

print("✓ Spark session initialized successfully")
print(f"Spark Version: {spark.version}")

## Section 2: Load Source Tables
Load SalesPerson and Sales tables from CSV or data source

In [ ]:
# Option 1: Load from CSV files
csv_path_sales_person = "/path/to/SalesPerson.csv"
csv_path_sales = "/path/to/Sales.csv"

# Load SalesPerson table
df_sales_person = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(csv_path_sales_person)

# Load Sales table
df_sales = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(csv_path_sales)

print("=== SalesPerson Table ===")
print(f"Schema:")
df_sales_person.printSchema()
print(f"Row count: {df_sales_person.count()}")
print("\nSample data:")
df_sales_person.show(3)

print("\n=== Sales Table ===")
print(f"Schema:")
df_sales.printSchema()
print(f"Row count: {df_sales.count()}")
print("\nSample data:")
df_sales.show(3)

## Section 3: Merge Tables Using Join Operations
Combine SalesPerson and Sales tables based on EmployeeID field

In [ ]:
# Define join condition
# Assume both tables have 'EmployeeID' column as the join key
join_condition = (df_sales_person["EmployeeID"] == df_sales["EmployeeID"])

# OPTION 1: INNER JOIN (keep only matching records from both tables)
df_combined_inner = df_sales_person.join(
    df_sales,
    on=join_condition,
    how="inner"
)

# OPTION 2: LEFT JOIN (keep all from SalesPerson, matching sales data)
df_combined_left = df_sales_person.join(
    df_sales,
    on=join_condition,
    how="left"
)

# Use INNER JOIN for this example
df_combined = df_combined_inner

print("=== Join Results ===")
print(f"Inner Join row count: {df_combined.count()}")
print(f"Schema after join:")
df_combined.printSchema()
print("\nSample combined data:")
df_combined.show(5, truncate=False)

## Section 4: Create New Combined Table with Column Mapping

Now we'll select and rename columns to create a clean, production-ready combined table.

In [ ]:
# Select and rename columns from the combined table
df_final = df_combined.select(
    col("EmployeeID"),
    col("Name").alias("EmployeeName"),
    col("Title"),
    col("Region"),
    col("SalesAmount"),
    col("OrderDate"),
    col("Quantity"),
    # Add calculated columns
    round(col("SalesAmount") / col("Quantity"), 2).alias("UnitPrice"),
    year(col("OrderDate")).alias("SalesYear"),
    month(col("OrderDate")).alias("SalesMonth")
)

print("=== Final Combined Table Schema ===")
df_final.printSchema()

print("\n=== Sample Data from Combined Table ===")
df_final.show(10, truncate=False)

print(f"\nTotal records in combined table: {df_final.count()}")
print(f"\nTotal sales amount: ${df_final.agg(sum('SalesAmount')).collect()[0][0]:,.2f}")
print(f"Average sale value: ${df_final.agg(avg('SalesAmount')).collect()[0][0]:,.2f}")

## Section 5: Define DAX Measures for Analytics (Power BI Integration)

DAX (Data Analysis Expressions) measures enable powerful analytics in Power BI. Below are example measures you can define in Power BI using this combined table.

### Common DAX Measures for Sales Analytics:

In [ ]:
# DAX Measures (copy and paste into Power BI)
dax_measures = """
DAX MEASURE DEFINITIONS FOR POWER BI:
=====================================

1. Total Sales Amount
   = SUM(CombinedTable[SalesAmount])

2. Total Quantity Sold
   = SUM(CombinedTable[Quantity])

3. Average Sale Value
   = AVERAGE(CombinedTable[SalesAmount])

4. Sales Transaction Count
   = COUNTROWS(CombinedTable)

5. Number of Unique Employees
   = DISTINCTCOUNT(CombinedTable[EmployeeID])

6. Sales Per Employee
   = DIVIDE([Total Sales Amount], [Number of Unique Employees])

7. Average Transaction Size
   = DIVIDE([Total Sales Amount], [Sales Transaction Count])

8. Max Sale Amount
   = MAXX(VALUES(CombinedTable[SalesAmount]), [SalesAmount])

9. Min Sale Amount
   = MINX(VALUES(CombinedTable[SalesAmount]), [SalesAmount])

10. Sales Growth (YoY)
    = VAR CurrentYear = YEAR(TODAY())
      VAR PriorYear = CurrentYear - 1
      VAR CurrentYearSales = CALCULATE([Total Sales Amount], 
                                       YEAR(CombinedTable[OrderDate]) = CurrentYear)
      VAR PriorYearSales = CALCULATE([Total Sales Amount], 
                                     YEAR(CombinedTable[OrderDate]) = PriorYear)
      RETURN DIVIDE(CurrentYearSales - PriorYearSales, PriorYearSales)

11. Sales by Region (for Matrix visualization)
    = CALCULATE([Total Sales Amount], 
                ALLEXCEPT(CombinedTable, CombinedTable[Region]))

12. Sales by Employee (for detailed analysis)
    = CALCULATE([Total Sales Amount], 
                ALLEXCEPT(CombinedTable, CombinedTable[EmployeeName]))

13. Top Performer This Year
    = VAR TopEmployee = TOPN(1, 
                             VALUES(CombinedTable[EmployeeName]), 
                             [Total Sales Amount], DESC)
      RETURN CONCATENATEX(TopEmployee, CombinedTable[EmployeeName], \", \")

14. Quarterly Sales Comparison
    = VAR CurrentQuarter = ROUNDUP(MONTH(TODAY())/3, 0)
      RETURN CALCULATE([Total Sales Amount], 
                       ROUNDUP(MONTH(CombinedTable[OrderDate])/3, 0) = CurrentQuarter)

15. Sales Pipeline Health (% of Target)
    = VAR SalesTarget = 100000
      VAR ActualSales = [Total Sales Amount]
      RETURN DIVIDE(ActualSales, SalesTarget)
"""

print(dax_measures)

## Section 6: Save Combined Table to Production Format

Save the combined table to Parquet or Delta Lake format for long-term storage and analytics.

In [ ]:
# Define output paths
output_path_parquet = "/mnt/data/combined_sales_table.parquet"
output_path_delta = "/mnt/data/combined_sales_table_delta"

# OPTION 1: Save to Parquet (widely supported, compressed, columnar format)
print("Saving to Parquet format...")
df_final.write \
    .mode("overwrite") \
    .parquet(output_path_parquet)
print(f"✓ Parquet file saved to: {output_path_parquet}")

# OPTION 2: Save to Delta Lake (ACID transactions, time travel, data quality)
print("\nSaving to Delta Lake format...")
df_final.write \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .format("delta") \
    .save(output_path_delta)
print(f"✓ Delta Lake saved to: {output_path_delta}")

# Verify saved data
print("\n=== Verification ===")
df_parquet = spark.read.parquet(output_path_parquet)
print(f"Parquet row count: {df_parquet.count()}")

df_delta = spark.read.format("delta").load(output_path_delta)
print(f"Delta row count: {df_delta.count()}")
print("✓ Both formats saved successfully!")

## Section 7: Validate Combined Table Quality

Run data quality checks to ensure the combined table is production-ready.

In [ ]:
# Data Quality Validation
print("=== DATA QUALITY VALIDATION ===\n")

# Check 1: Row count
row_count = df_final.count()
assert row_count > 0, "ERROR: Combined table is empty!"
print(f"✓ Row count: {row_count:,} records")

# Check 2: NULL values in critical columns
critical_cols = ["EmployeeID", "EmployeeName", "SalesAmount"]
for col_name in critical_cols:
    null_count = df_final.filter(col(col_name).isNull()).count()
    assert null_count == 0, f"ERROR: Found {null_count} NULLs in {col_name}"
    print(f"✓ No NULLs in {col_name}")

# Check 3: Sales amount validation
min_sales = df_final.agg(min("SalesAmount")).collect()[0][0]
max_sales = df_final.agg(max("SalesAmount")).collect()[0][0]
assert min_sales >= 0, "ERROR: Negative sales amounts found!"
assert max_sales > 0, "ERROR: No positive sales found!"
print(f"✓ Sales amount range: ${min_sales:,.2f} - ${max_sales:,.2f}")

# Check 4: Date validation
min_date = df_final.agg(min("OrderDate")).collect()[0][0]
max_date = df_final.agg(max("OrderDate")).collect()[0][0]
print(f"✓ Date range: {min_date} to {max_date}")

# Check 5: Unique employee count
unique_employees = df_final.select("EmployeeID").distinct().count()
print(f"✓ Unique employees: {unique_employees}")

# Check 6: Duplicate records
total_rows = df_final.count()
distinct_rows = df_final.select("EmployeeID", "OrderDate", "SalesAmount").distinct().count()
duplicates = total_rows - distinct_rows
print(f"✓ Total rows: {total_rows}, Unique combinations: {distinct_rows}")
if duplicates > 0:
    print(f"  Note: {duplicates} potential duplicate rows (verify if expected)")

# Check 7: Data types
print(f"\n✓ Data types validation:")
for field in df_final.schema.fields:
    print(f"  - {field.name}: {field.dataType}")

# Summary
print("\n" + "="*50)
print("✓✓✓ ALL VALIDATION CHECKS PASSED ✓✓✓")
print("Combined table is ready for analytics!")
print("="*50)